# Import

In [1]:
import time
import re
from datetime import timedelta
import datasets
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import wordnet, stopwords
from nltk.stem import WordNetLemmatizer

# nltk.download('popular')

lemmatizer = WordNetLemmatizer()
wordnet_map = {"N":wordnet.NOUN, "V":wordnet.VERB, "J":wordnet.ADJ, "R":wordnet.ADV}

# data

In [2]:
imdb_set = datasets.load_dataset('imdb')
print(imdb_set)

Found cached dataset imdb (/home/kd/.cache/huggingface/datasets/imdb/plain_text/1.0.0/d613c88cf8fa3bab83b4ded3713f1f74830d1100e171db75bbddb80b3345c9c0)


  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


# data sample

In [3]:
imdb_set['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

# prepare data

In [4]:
imdb_trn_df = pd.DataFrame(imdb_set['train'])
imdb_tst_df = pd.DataFrame(imdb_set['test'])

In [5]:
# clean & prepare data
def clean_sentences(line):
    
    # line=re.sub('<.*?>','',line) # removing html tags
    
    #removing contractions
    line=re.sub("isn't",'is not',line)
    line=re.sub("he's",'he is',line)
    line=re.sub("wasn't",'was not',line)
    line=re.sub("there's",'there is',line)
    line=re.sub("couldn't",'could not',line)
    line=re.sub("won't",'will not',line)
    line=re.sub("they're",'they are',line)
    line=re.sub("she's",'she is',line)
    line=re.sub("There's",'there is',line)
    line=re.sub("wouldn't",'would not',line)
    line=re.sub("haven't",'have not',line)
    line=re.sub("That's",'That is',line)
    line=re.sub("you've",'you have',line)
    line=re.sub("He's",'He is',line)
    line=re.sub("what's",'what is',line)
    line=re.sub("weren't",'were not',line)
    line=re.sub("we're",'we are',line)
    line=re.sub("hasn't",'has not',line)
    line=re.sub("you'd",'you would',line)
    line=re.sub("shouldn't",'should not',line)
    line=re.sub("let's",'let us',line)
    line=re.sub("they've",'they have',line)
    line=re.sub("You'll",'You will',line)
    line=re.sub("i'm",'i am',line)
    line=re.sub("we've",'we have',line)
    line=re.sub("it's",'it is',line)
    line=re.sub("don't",'do not',line)
    line=re.sub("that´s",'that is',line)
    line=re.sub("I´m",'I am',line)
    line=re.sub("it’s",'it is',line)
    line=re.sub("she´s",'she is',line)
    line=re.sub("he’s'",'he is',line)
    line=re.sub('I’m','I am',line)
    line=re.sub('I’d','I did',line)
    line=re.sub("he’s'",'he is',line)
    line=re.sub('there’s','there is',line)
    
    #special characters and emojis
    line=re.sub('\x91The','The',line)
    line=re.sub('\x97','',line)
    line=re.sub('\x84The','The',line)
    line=re.sub('\uf0b7','',line)
    line=re.sub('¡¨','',line)
    line=re.sub('\x95','',line)
    line=re.sub('\x8ei\x9eek','',line)
    line=re.sub('\xad','',line)
    line=re.sub('\x84bubble','bubble',line)
    
    # remove concated words
    line=re.sub('trivialBoring','trivial Boring',line)
    line=re.sub('Justforkix','Just for kix',line)
    line=re.sub('Nightbeast','Night beast',line)
    line=re.sub('DEATHTRAP','Death Trap',line)
    line=re.sub('CitizenX','Citizen X',line)
    line=re.sub('10Rated','10 Rated',line)
    line=re.sub('_The','_ The',line)
    line=re.sub('1Sound','1 Sound',line)
    line=re.sub('blahblahblahblahblahblahblahblahblahblahblahblahblahblahblahblahblahblah','blah blah',line)
    line=re.sub('ResidentHazard','Resident Hazard',line)
    line=re.sub('iameracing','i am racing',line)
    line=re.sub('BLACKSNAKE','Black Snake',line)
    line=re.sub('DEATHSTALKER','Death Stalker',line)
    line=re.sub('_is_','is',line)
    line=re.sub('10Fans','10 Fans',line)
    line=re.sub('Yellowcoat','Yellow coat',line)
    line=re.sub('Spiderbabe','Spider babe',line)
    line=re.sub('Frightworld','Fright world',line)

    return line


def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()


def remove_stopwords(text):
    words = set(stopwords.words("english")) # conversion into set for fast searching !!!
    example_review = [word for word in text if word not in words]              
    return text


def lemmatize_words(text):
    text = strip_html(text)
    pos_tagged_text = nltk.pos_tag(text.split())
    return " ".join([lemmatizer.lemmatize(word, wordnet_map.get(pos[0], wordnet.NOUN)) for word, pos in pos_tagged_text])
    

def prepare_data(data, name):
    
    line = '='*60
    print(line)
    print(f'Handling data: {name}')
    print(line)
    print('descriptive stats:')
    print(data['text'].apply(lambda x: len(x)).describe())
    print(line)

    # check for HTML tags present in the text
    print('HTML tags found: ', data['text'].loc[data[data['text'].apply(lambda x: '<br' in x)].index].count())
    print(line)

    # remove HTML tags
    data['text'] = data['text'].apply(lambda x: strip_html(x))
    print('HTML tags found (after removing): ', data['text'].loc[data[data['text'].apply(lambda x: '<br' in x)].index].count())
    print(line)

    # remove stop words
    print('removing stopwords..')
    data['text'] = data['text'].apply(lambda x: remove_stopwords(x))
    print(line)

    # simplify contractions, special chars, emojis & concated words
    print('Cleanig further by replacing contractions, special chars, emojis & concated words..')
    data['text'] = data['text'].apply(lambda x: clean_sentences(x))
    print(line)

    # lower casing & lemmatize text
    print('Performed lemmatization in', end=' ')
    t0 = time.time()
    data['text'] = data['text'].apply(lambda x: lemmatize_words(x.lower()))
    print(str(timedelta(seconds=time.time()-t0)))

    return data

# combine ***IMDB*** & ***Rotten tomatoes*** dataset




In [6]:
train_df = prepare_data(imdb_trn_df, 'IMDB train')
test_df  = prepare_data(imdb_tst_df, 'IMDB test')

Handling data: IMDB train
descriptive stats:
count    25000.00000
mean      1325.06964
std       1003.13367
min         52.00000
25%        702.00000
50%        979.00000
75%       1614.00000
max      13704.00000
Name: text, dtype: float64
HTML tags found:  14665


/home/kd/workspace/nltk-env/lib/python3.8/site-packages/bs4/__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


HTML tags found (after removing):  0
removing stopwords..
Cleanig further by replacing contractions, special chars, emojis & concated words..
Performed lemmatization in 0:02:31.381734
Handling data: IMDB test
descriptive stats:
count    25000.00000
mean      1293.79240
std        975.90776
min         32.00000
25%        696.00000
50%        962.00000
75%       1572.00000
max      12988.00000
Name: text, dtype: float64
HTML tags found:  14535


/home/kd/workspace/nltk-env/lib/python3.8/site-packages/bs4/__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


HTML tags found (after removing):  0
removing stopwords..
Cleanig further by replacing contractions, special chars, emojis & concated words..
Performed lemmatization in 0:02:25.302371


# train & Test dataset

In [7]:
train_df.loc[train_df['label'] == 1, 'label'] = 2
train_df.loc[train_df['label'] == 0, 'label'] = 1

train_df

,text,label
0,i rent i be curious-yellow from my video store...,1
1,"""i be curious: yellow"" be a risible and preten...",1
2,if only to avoid make this type of film in the...,1
3,this film be probably inspire by godard's masc...,1
4,"oh, brother...after hear about this ridiculous...",1
...,...,...
24995,a hit at the time but now well categorise a an...,2
24996,i love this movie like no other. another time ...,2
24997,this film and it be sequel barry mckenzie hold...,2
24998,'the adventure of barry mckenzie' start life a...,2


In [8]:
test_df.loc[test_df['label'] == 1, 'label'] = 2
test_df.loc[test_df['label'] == 0, 'label'] = 1

test_df

,text,label
0,i love sci-fi and be willing to put up with a ...,1
1,"worth the entertainment value of a rental, esp...",1
2,it a totally average film with a few semi-alri...,1
3,star rating: ***** saturday night **** friday ...,1
4,"first off let me say, if you have not enjoy a ...",1
...,...,...
24995,just get around to see monster man yesterday. ...,2
24996,i get this a part of a competition prize. i wa...,2
24997,i get monster man in a box set of three film w...,2
24998,"five minute in, i start to feel how naff this ...",2


# Vectorize (using PyTorch)

## Attempt-1

### Import

In [9]:
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split
import torchtext
from torchtext.data import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

### Build Vocab

In [10]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(dataset):
    for idx, row in dataset.iterrows():
            yield tokenizer(row['text'])

vocab = build_vocab_from_iterator(yield_tokens(train_df), specials=["<UNK>"])
vocab.set_default_index(vocab["<UNK>"])

print(len(vocab.get_itos()))

96984


In [11]:
text_pipeline = lambda x: vocab(tokenizer(x))
label_pipeline = lambda x: int(x) - 1

print(text_pipeline('here is an example of god'))
print(label_pipeline("2"))

[128, 225, 37, 426, 7, 534]
1


### Dataloader

In [12]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.data = df   

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text, label = self.data.iloc[index]
        # sample = {"Text": text, "Class": label}
        return text, label

In [13]:
def collate_batch(batch):
     label_list, text_list, offsets = [], [], [0]
     for (_text, _label) in batch:
          label_list.append(label_pipeline(_label))
          processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
          text_list.append(processed_text)
          offsets.append(processed_text.size(0))
     label_list = torch.tensor(label_list, dtype=torch.int64)
     offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
     text_list = torch.cat(text_list)
     return label_list, text_list, offsets

### Model

In [14]:
class TextClassificationModel(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_class):
        super(TextClassificationModel, self).__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=True)
        self.fc = nn.Linear(embed_dim, num_class)
        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        return self.fc(embedded)

In [15]:
num_class = 2
vocab_size = len(vocab)
embed_size = 64
model = TextClassificationModel(vocab_size, embed_size, num_class).to(device)

### Hypyerparameters

In [16]:
# Hyperparameters
EPOCHS = 10 # epoch
LR = 5  # learning rate
BATCH_SIZE = 32 # batch size for training

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)
total_accu = None


### Training & Validation

In [17]:
def train(dataloader):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        text, label, offsets = text.to(device), label.to(device), offsets.to(device)
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)
        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print('| epoch {:3d} | {:5d}/{:5d} batches '
                  '| accuracy {:8.3f}'.format(epoch, idx, len(dataloader),
                                              total_acc/total_count))
            total_acc, total_count = 0, 0
            start_time = time.time()

def evaluate(dataloader):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            text, label, offsets = text.to(device), label.to(device), offsets.to(device)
            predicted_label = model(text, offsets)
            loss = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)
    return total_acc/total_count


In [18]:
train_ds = TextDataset(train_df)
test_ds = TextDataset(test_df)

num_train = int(len(train_ds) * 0.90)
split_train_, split_valid_ = random_split(train_ds, [num_train, len(train_ds) - num_train])

train_dataloader = DataLoader(split_train_, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(split_valid_, batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_batch)
test_dataloader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                             shuffle=True, collate_fn=collate_batch)


for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    train(train_dataloader)
    accu_val = evaluate(valid_dataloader)
    if total_accu is not None and total_accu > accu_val:
      scheduler.step()
    else:
       total_accu = accu_val
    print('-' * 59)
    print('| end of epoch {:3d} | time: {:5.2f}s | '
          'valid accuracy {:8.3f} '.format(epoch,
                                           time.time() - epoch_start_time,
                                           accu_val))
    print('-' * 59)

| epoch   1 |   500/  704 batches | accuracy    0.670
-----------------------------------------------------------
| end of epoch   1 | time:  6.60s | valid accuracy    0.753 
-----------------------------------------------------------
| epoch   2 |   500/  704 batches | accuracy    0.801
-----------------------------------------------------------
| end of epoch   2 | time:  5.29s | valid accuracy    0.829 
-----------------------------------------------------------
| epoch   3 |   500/  704 batches | accuracy    0.830
-----------------------------------------------------------
| end of epoch   3 | time:  5.04s | valid accuracy    0.811 
-----------------------------------------------------------
| epoch   4 |   500/  704 batches | accuracy    0.873
-----------------------------------------------------------
| end of epoch   4 | time:  4.97s | valid accuracy    0.860 
-----------------------------------------------------------
| epoch   5 |   500/  704 batches | accuracy    0.875
------

### Testing

In [19]:
print('Checking the results of test dataset.')
accu_test = evaluate(test_dataloader)
print('test accuracy {:8.3f}'.format(accu_test))

Checking the results of test dataset.
test accuracy    0.862
